## Setup

In [31]:
# Imports

from pathlib import Path

import geopandas as gpd
import numpy as np
from shapely.geometry import Point


In [65]:
# -----------------------------------------
# Paths & CRS
# -----------------------------------------

soils_geojson = Path(
    "/home/zagreus/leap-hackathon-flushing/data/processed/rain-gardens/nyc_native_soils_for_rain_gardens.geojson"
)
slope_geojson = Path(
    "/home/zagreus/leap-hackathon-flushing/data/processed/rain-gardens/nyc_tracts_slope.geojson"
)

# Common working CRS (ft) for NYC
TARGET_CRS = "EPSG:2263"  # NAD83 / New York Long Island (ftUS)

# -----------------------------------------
# Read and reproject data
# -----------------------------------------

gdf_soils = gpd.read_file(soils_geojson)
gdf_slope = gpd.read_file(slope_geojson)

# Reproject both layers to common CRS
gdf_soils = gdf_soils.to_crs(TARGET_CRS)
gdf_slope = gdf_slope.to_crs(TARGET_CRS)

# -----------------------------------------
# Hydrologic soil group → infiltration
# -----------------------------------------

# Base infiltration map (in/hr) for single-letter HSG
BASE_HSG_INFILTRATION_INHR = {
    "A": 0.45,
    "B": 0.225,
    "C": 0.10,
    "D": 0.025,
}


def hsg_to_infiltration(
    hsg: str | None,
    strategy: str = "conservative",
) -> float | np.nan:
    """
    Map hydrologic soil group (including dual groups like A/D) to an
    infiltration rate in in/hr using the chosen strategy.

    strategy options:
        - "conservative": dual groups -> D
        - "optimistic":   dual groups -> first letter (A/B/C)
        - "average":      dual groups -> mean of drained and undrained rates
    """
    if hsg is None or (isinstance(hsg, float) and np.isnan(hsg)):
        return np.nan

    hsg = str(hsg).strip().upper()

    # Single-letter group
    if "/" not in hsg:
        return BASE_HSG_INFILTRATION_INHR.get(hsg, np.nan)

    # Dual group, e.g. "A/D", "B/D"
    parts = [p.strip() for p in hsg.split("/") if p.strip()]
    if len(parts) != 2:
        return np.nan

    first, second = parts

    first_rate = BASE_HSG_INFILTRATION_INHR.get(first)
    second_rate = BASE_HSG_INFILTRATION_INHR.get(second)

    if first_rate is None or second_rate is None:
        return np.nan

    if strategy == "conservative":
        # Assume undrained → use second letter (usually D)
        return second_rate
    elif strategy == "optimistic":
        # Assume drained → use first letter (A/B/C)
        return first_rate
    elif strategy == "average":
        # Average of drained and undrained behavior
        return 0.5 * (first_rate + second_rate)

    raise ValueError(f"Unknown strategy: {strategy}")


# Choose model strategy here
HSG_STRATEGY = "conservative"  # "optimistic" or "average" also allowed

# Map hsg_native to infiltration rates
gdf_soils["infiltration_inhr"] = gdf_soils["hsg_native"].apply(
    lambda val: hsg_to_infiltration(val, strategy=HSG_STRATEGY)
)

# Convert to ft/hr for later area/rainfall math
gdf_soils["infiltration_fthr"] = gdf_soils["infiltration_inhr"] / 12.0

# Optional quick check of mappings
_ = (
    gdf_soils[["hsg_native", "infiltration_inhr"]]
    .drop_duplicates()
    .sort_values("hsg_native")
)

# -----------------------------------------
# Slope → efficiency factor
# -----------------------------------------

SLOPE_THRESHOLD = 5.0  # %
SLOPE_DECAY = 0.18  # 1 / %


def slope_efficiency(slope_pct: float) -> float:
    """
    Exponential decay in effectiveness above a threshold slope.
    Returns a dimensionless factor in (0, 1].
    """
    if slope_pct <= SLOPE_THRESHOLD:
        return 1.0
    return float(np.exp(-SLOPE_DECAY * (slope_pct - SLOPE_THRESHOLD)))


# NOTE: adjust "Slope" below if your slope column has a different name
gdf_slope["slope_factor"] = gdf_slope["Slope"].apply(slope_efficiency)

# -----------------------------------------
# Geometry helpers
# -----------------------------------------


def point_from_latlon(lat: float, lon: float, crs: str = "EPSG:4326"):
    """
    Create a shapely Point from lat/lon and project it to the target CRS.
    """
    return gpd.GeoSeries([Point(lon, lat)], crs=crs).to_crs(TARGET_CRS).iloc[0]


def area_weighted_mean_for_polygon(
    polygon,
    gdf: gpd.GeoDataFrame,
    value_col: str,
) -> float | np.nan:
    """
    Compute the area-weighted mean of `value_col` from `gdf`
    over the intersection with `polygon`.

    Parameters
    ----------
    polygon : shapely geometry
        Target polygon (must be in same CRS as gdf)
    gdf : GeoDataFrame
        Source polygons with attribute `value_col`
    value_col : str
        Column name containing numeric values

    Returns
    -------
    float or np.nan
        Area-weighted mean value
    """
    # Wrap polygon in GeoDataFrame
    gdf_poly = gpd.GeoDataFrame(
        {"geometry": [polygon]},
        crs=gdf.crs,
    )

    # Spatial intersection
    inter = gpd.overlay(gdf_poly, gdf, how="intersection")

    if inter.empty:
        return np.nan

    # Compute intersection areas
    inter["area"] = inter.geometry.area

    # Drop rows with missing values
    inter = inter.dropna(subset=[value_col])

    if inter.empty:
        return np.nan

    return (inter[value_col] * inter["area"]).sum() / inter["area"].sum()


def effective_infiltration_at_point(
    lat: float,
    lon: float,
    gdf_soils: gpd.GeoDataFrame,
    gdf_slope: gpd.GeoDataFrame,
    buffer_ft: float = 50.0,
) -> float:
    """
    Compute effective infiltration (in/hr) at a given lat/lon by
    combining soil infiltration and slope factor within a small buffer.
    """
    point = point_from_latlon(lat, lon)

    # Small buffer for robustness / local averaging
    geom = point.buffer(buffer_ft)

    infil = area_weighted_mean_for_polygon(geom, gdf_soils, "infiltration_inhr")
    slope_factor = area_weighted_mean_for_polygon(geom, gdf_slope, "slope_factor")

    if np.isnan(infil) or np.isnan(slope_factor):
        raise ValueError("Could not determine soil or slope at this location.")

    return float(infil * slope_factor)


# -----------------------------------------
# Bucket model: rainfall → gallons saved
# -----------------------------------------

FT3_TO_GALLONS = 7.48052

DRAINAGE_RATIO = 5.0  # A_drain = 10 × A_garden
PONDING_DEPTH_IN = 6.0  # inches
RUNOFF_COEFF = 0.9  # mostly impervious


def rain_garden_gallons_saved(
    rain_in_per_hr,
    garden_area_ft2: float,
    effective_infiltration_inhr: float,
) -> float:
    """
    Compute total gallons of runoff captured by a rain garden over time.

    Parameters
    ----------
    rain_in_per_hr : array-like
        Hourly rainfall rates [in/hr]
    garden_area_ft2 : float
        Rain garden surface area [ft²]
    effective_infiltration_inhr : float
        Soil × slope adjusted infiltration rate [in/hr]

    Returns
    -------
    float
        Total gallons of runoff captured (not overflowed).
    """

    rain_arr = np.asarray(rain_in_per_hr, dtype=float)

    # --- geometry ---
    A_g = float(garden_area_ft2)
    A_d = DRAINAGE_RATIO * A_g

    # --- infiltration ---
    infil_fthr = effective_infiltration_inhr / 12.0
    infil_capacity_ft3_per_hr = infil_fthr * A_g

    # --- storage ---
    ponding_depth_ft = PONDING_DEPTH_IN / 12.0
    storage_max_ft3 = A_g * ponding_depth_ft
    storage_ft3 = 0.0

    total_captured_ft3 = 0.0

    for R_in in rain_arr:
        # Treat NaN or negative as 0
        if np.isnan(R_in) or R_in < 0:
            R_in = 0.0

        # rainfall this hour
        R_ft = R_in / 12.0

        # runoff inflow
        Qin_ft3 = RUNOFF_COEFF * R_ft * A_d

        # available water
        available_ft3 = storage_ft3 + Qin_ft3

        # infiltration (cannot exceed available water)
        infiltrated_ft3 = min(infil_capacity_ft3_per_hr, available_ft3)

        # provisional storage
        storage_prov_ft3 = available_ft3 - infiltrated_ft3

        # overflow beyond max storage
        overflow_ft3 = max(0.0, storage_prov_ft3 - storage_max_ft3)

        # final storage
        storage_ft3 = storage_prov_ft3 - overflow_ft3
        storage_ft3 = max(0.0, storage_ft3)  # numerical safety

        # captured = inflow minus overflow
        captured_ft3 = Qin_ft3 - overflow_ft3
        total_captured_ft3 += captured_ft3

    return total_captured_ft3 * FT3_TO_GALLONS


In [86]:
assets_geojson = Path(
    "/home/zagreus/leap-hackathon-flushing/data/processed/gi-surface-queens.geojson"
)

gdf_assets = gpd.read_file(assets_geojson)

gdf_rain_gardens = gdf_assets[gdf_assets["asset_type"] == "Rain Garden"].copy()

gdf_rain_gardens["lon"] = gdf_rain_gardens.geometry.x
gdf_rain_gardens["lat"] = gdf_rain_gardens.geometry.y

results = []

for idx, row in gdf_rain_gardens.iterrows():
    area_ft2 = row["asset_area"]

    try:
        eff_infil = effective_infiltration_at_point(
            lat=row["lat"],  # degrees
            lon=row["lon"],  # degrees
            gdf_soils=gdf_soils,
            gdf_slope=gdf_slope,
            buffer_ft=50.0,
        )

        gallons = rain_garden_gallons_saved(
            rain_in_per_hr=rain_test,
            garden_area_ft2=area_ft2,
            effective_infiltration_inhr=eff_infil,
        )

    except Exception as e:
        eff_infil = np.nan
        gallons = np.nan

    results.append(
        {
            "asset_id": idx,
            "asset_area_ft2": area_ft2,
            "effective_infiltration_inhr": eff_infil,
            "annual_gallons_saved": gallons,
        }
    )


Skipping field time_const: unsupported OGR type: 10


In [87]:
gdf_results = gdf_rain_gardens.copy()
gdf_results = gdf_results.join(
    gpd.GeoDataFrame(results).set_index("asset_id"),
    how="left",
)


In [79]:
# --- quick smoke test (toy example) ---

# Fake hourly rain: 10 storms of 0.25 in/hr lasting 3 hours
hours = 365 * 24
rain_test = np.zeros(hours)
storm_hours = np.random.choice(hours, size=10 * 3, replace=False)
rain_test[storm_hours] = 0.25

# Pick a lat/lon in Queens roughly
lat_test, lon_test = 40.74305135141636, -73.81398517025595
eff_infil_test = effective_infiltration_at_point(
    lat_test, lon_test, gdf_soils, gdf_slope
)

gallons_test = rain_garden_gallons_saved(
    rain_test,
    garden_area_ft2=250.0,
    effective_infiltration_inhr=eff_infil_test,
)

print(
    f"Test run: effective infiltration ≈ {eff_infil_test:.3f} in/hr, "
    f"gallons saved ≈ {gallons_test:,.0f}"
)


Test run: effective infiltration ≈ 0.225 in/hr, gallons saved ≈ 5,260
